In [0]:
%sql
-- ゴールド層のテーブルを削除
DROP TABLE IF EXISTS cf_mom_factory_dev.gold.gld_api_facility_maintenance;
DROP TABLE IF EXISTS cf_mom_factory_dev.gold.gld_api_operator_status;
DROP TABLE IF EXISTS cf_mom_factory_dev.gold.gld_api_vehicle_trace;
DROP TABLE IF EXISTS cf_mom_factory_dev.gold.gld_bi_mom_factory;

-- シルバー層のテーブルを削除
DROP TABLE IF EXISTS cf_mom_factory_dev.silver.slv_mom_factory;

-- ブロンズ層のテーブルを削除
DROP TABLE IF EXISTS cf_mom_factory_dev.bronze.brz_mom_factory;

In [0]:
%sql
select * from cf_mom_factory_dev.silver.slv_mom_factory
where log_id = 'F01-L0019';

In [0]:
%sql
select * from cf_mom_factory_dev_matsuot.gold.gld_api_operator_status
where vin = 'VIN-F04-00001';

In [0]:
%sql
SELECT * FROM cf_engineering_dev_matsuot.master.m_production_standard_sp;

In [0]:
%sql
DROP TABLE IF EXISTS cf_engineering_dev_matsuot.master.m_production_standard;

CREATE TABLE cf_engineering_dev_matsuot.master.m_production_standard (
    car_model_code STRING NOT NULL,
    process_code STRING NOT NULL,
    design_version STRING NOT NULL,
    bom_id STRING,
    target_cycle_time DOUBLE,
    _input_file_path STRING,
    _processed_timestamp TIMESTAMP
)
USING delta;

ALTER TABLE cf_engineering_dev_matsuot.master.m_production_standard 
ADD CONSTRAINT m_production_standard_pk PRIMARY KEY (car_model_code, process_code, design_version);

In [0]:
%sql
select max(timestamp) as time from cf_mom_factory_dev_matsuot.bronze.brz_mom_factory;

In [0]:
%sql
SELECT 
  _processed_timestamp,
  
  CASE 
    WHEN _rescued_data IS NOT NULL THEN 'Auto Loader rescued row (スキーマ構造の破壊を検知)'
    WHEN log_id IS NULL THEN 'log_id is null (必須キーの欠損)'
    WHEN to_timestamp(timestamp, 'yyyy/M/d HH:mm') IS NULL THEN 'timestamp parse failed (日付フォーマット不正)'
    ELSE '【テスト用擬似エラー】cycle_time_secの値が製造閾値（100秒）をわずかに超過しています' 
  END AS mock_failure_reason,

  ai_query(
    -- ★見つけていただいた正しいモデル名に差し替えました！
    'databricks-meta-llama-3-3-70b-instruct', 
    
    concat(
      'あなたは工場のデータ基盤を監視するAIエンジニアです。以下のDatabricksのデータ異常ログについて、現場のオペレーターや保全担当者が「何が起きていて、どこを確認すべきか」の対策を、日本語で詳細に簡潔に解説してください。\n\n',
      '■検知されたエラー：', 
      CASE 
        WHEN _rescued_data IS NOT NULL THEN 'Auto Loader rescued row (スキーマ破壊)'
        WHEN log_id IS NULL THEN 'log_id is null (主キー欠損)'
        WHEN to_timestamp(timestamp, 'yyyy/M/d HH:mm') IS NULL THEN 'timestamp parse failed (日付パースエラー)'
        ELSE '【テスト用擬似エラー】cycle_time_secの値が製造閾値（100秒）をわずかに超過しています' 
      END,
      '\n■発生データの詳細（JSON）：', 
      to_json(struct(
        log_id, vin, timestamp, factory_id, line_id, process_code, 
        car_model_code, process_result_code, cycle_time_sec, defect_code, 
        operator_id, parts_lot_no, recipe_id, `4m_changed_flg`, sequence_no
      ))
    )
  ) AS ai_analysis

FROM 
  cf_mom_factory_dev_matsuot.bronze.brz_mom_factory
LIMIT 5;

In [0]:
%sql
DROP CATALOG IF EXISTS cf_engineering_dev_matsuot CASCADE;
-- DROP CATALOG IF EXISTS cf_engineering_dev CASCADE;

In [0]:
%sql
-- 🚨 古いストリーミング属性の器を物理的に完全消去します
DROP TABLE cf_mom_factory_dev_matsuot.gold.gld_qt_mom_factory;